In [ ]:

import os
import io
import zipfile
import requests
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print('Imports Tmam')

In [ ]:
DATA_ZIP_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

r = requests.get(DATA_ZIP_URL)
if r.status_code != 200:
    raise RuntimeError(f"Failed to download dataset: {r.status_code}")

z = zipfile.ZipFile(io.BytesIO(r.content))
print('Files in ZIP:', z.namelist())

with z.open('SMSSpamCollection.txt') as f:
    df = pd.read_csv(f, sep='\t', header=None, names=['label', 'text'])

print('Loaded dataset with', len(df), 'rows')

df.head()

In [ ]:

print(df['label'].value_counts())
print('\nSample texts:')
for i, row in df.sample(5, random_state=42).iterrows():
    print('-', row['label'], ':', row['text'][:120])

print('\nText length stats:')
df['length'] = df['text'].str.len()
print(df['length'].describe())

In [ ]:
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

df['text_clean'] = df['text'].str.lower()

df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)

print('After preprocessing:', len(df))
df.head()

In [ ]:
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1,2))
X = vectorizer.fit_transform(df['text_clean'])
y = df['label_num'].values

print('Feature matrix shape:', X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = MultinomialNB()
model.fit(X_train, y_train)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"Test accuracy: {acc:.4f}")
print('\nClassification report:')
print(classification_report(y_test, preds, target_names=['ham','spam']))

In [ ]:
base_dir = os.path.dirname(os.path.abspath(__file__))
model_api_dir = os.path.join(base_dir, 'model_api')
os.makedirs(model_api_dir, exist_ok=True)

vec_path = os.path.join(model_api_dir, 'vectorizer.pkl')
model_path = os.path.join(model_api_dir, 'spam_model.pkl')

joblib.dump(vectorizer, vec_path)
joblib.dump(model, model_path)

print('Saved vectorizer to:', vec_path)
print('Saved model to:', model_path)

In [ ]:
loaded_vectorizer = joblib.load(vec_path)
loaded_model = joblib.load(model_path)

examples = [
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005.",
    "Are you coming to the meeting later?",
    "Congratulations! You have been selected for a $1000 Walmart gift card."
]

X_ex = loaded_vectorizer.transform([e.lower() for e in examples])
print('Predictions (1=spam,0=ham):', loaded_model.predict(X_ex).tolist())